In [ ]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report

In [ ]:
FEATURE_COLUMNS = [
    "degree",
    "betweenness centrality",
    "local clustering",
    "avg neighbour degree"
]

metadata = pd.read_csv("metadata.csv")

label_map = {"AD": 0, "CN": 1}

subject_to_label = dict(
    zip(
        metadata["subject_id"],
        metadata["group"].map(
            label_map
        )
    )
)

def load_features(path):
    rows = []
    with open(path, "r") as f:
        lines = f.readlines()

    header = lines[0].strip().split(",")
    header = [h.strip().lower() for h in header[:4]]

    for line in lines[1:]:
        values = line.strip().split(",")
        values = values[:14]
        rows.append(values)

    df = pd.DataFrame(rows, columns=header)
    return df.astype(np.float32).values.flatten()

X = []
y = []
for group_name, label in [("AD", 0), ("CN", 1)]:
    files = sorted(glob(os.path.join("./weighted_features", group_name, "*_adj.csv")))

    for f in files:
        features = load_features(f)
        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

feature_names = []
for roi in range(26):
    for feature in FEATURE_COLUMNS:
        feature_names.append(f"ROI_{roi+1}_{feature}")

feature_names = np.array(feature_names)

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_aucs = []
all_importances = []

feature_frequency = np.zeros(X.shape[1], dtype=int)

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    print(f"\nRunning Fold {fold}")

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    rf = RandomForestClassifier(n_estimators=4, max_features=69, random_state=42)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    probs = rf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test,preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, probs)

    fold_accuracies.append(acc)
    fold_precisions.append(prec)
    fold_recalls.append(rec)
    fold_aucs.append(auc)

    print(f"Accuracy = {acc:.4f}")
    print(f"Precision = {prec:.4f}")
    print(f"Recall = {rec:.4f}")
    print(f"ROC-AUC = {auc:.4f}")
    print(classification_report(y_test, preds, digits=4))

    importance = rf.feature_importances_
    importance_df = pd.DataFrame({
        "Feature_Name":
        feature_names,
        "Importance":
        importance,
        "Fold":
        fold
    })

    all_importances.append(importance_df)
    selected_features = np.where(importance > 0)[0]
    feature_frequency[selected_features] += 1

print("\nOverall Performance")
print(f"Mean Accuracy = {np.mean(fold_accuracies):.4f}")
print(f"Mean Precision = {np.mean(fold_precisions):.4f}")
print(f"Mean Recall = {np.mean(fold_recalls):.4f}")
print(f"Mean ROC-AUC = {np.mean(fold_aucs):.4f}")

all_importances = pd.concat(all_importances, ignore_index=True)

importance_summary = (all_importances.groupby("Feature_Name")["Importance"].agg(
    Mean_Importance="mean",
    Std_Importance="std",
    Max_Importance="max",
    Min_Importance="min").reset_index())

importance_summary["Frequency"] = feature_frequency

n_total_folds = 5 * 10

threshold = int(0.5 * n_total_folds)

stable_edges = importance_summary[importance_summary["Frequency"] > threshold]

stable_edges = stable_edges.sort_values(
    by=["Frequency", "Mean_Importance"],
    ascending=False
)

stable_edges.to_csv("stable_edges_rf.csv", index=False)
importance_summary.to_csv("all_feature_importances_rf.csv", index=False)

print("\nStable Edges")
print(stable_edges)

plt.figure(figsize=(10, 6))
metrics = [
    np.mean(fold_accuracies),
    np.mean(fold_precisions),
    np.mean(fold_recalls),
    np.mean(fold_aucs)
]
metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "ROC-AUC"

]
plt.bar(metric_names, metrics)
plt.ylabel("Score")
plt.title("Average RF Performance Across 50 CV Folds")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()